# SLC46A1 engraftment, step 4: double-substitution rescue at chicken 407

The single-substitution scan of step 2 covers every substitution at every
position one at a time, and no single substitution lifts the humanising
substitution at chicken 407 above the operative threshold. This notebook scores
pairs of substitutions at the positions most strongly coupled to the target in
the EVcouplings run, which is the part of the space step 2 cannot reach.

The scoring convention is the one step 2 uses: both rescuers are written into
the chicken sequence, the target position keeps its chicken residue, the full
sequence is passed through ESM-2 without masking, and the probability of the
human residue at the target position is read from the softmax at that position.
The same 190 single substitutions at the candidate positions are rescored here
and checked against the step 2 values, so the doubles and the singles are known
to be on one scale before they are compared.

It stops on fatal conditions instead of warning, checkpoints to Drive after
every batch, and resumes from the checkpoint if the runtime is lost. Run every
cell in order. Nothing outside the project folder on Drive is written.


In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'esm2_step4')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/coupling_guided_rescue.json',
            'data/primary_substitution_risk.tsv',
            'data/esm2_step2/rescue_scan.tsv',
            'structures/7BC6_to_human.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))
    print(line, flush=True)

log('start, project=%s' % PROJECT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]

if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
if len(chicken) != 473:
    raise SystemExit('FATAL chicken native length %d, expected 473' % len(chicken))

CHICKEN_SHA = hashlib.sha256(chicken.encode()).hexdigest()
log('sequences ok: human %d, chicken %d, chicken_sha256 %s'
    % (len(human), len(chicken), CHICKEN_SHA))


In [ ]:
THRESHOLD = 1e-3
N_POS = 10
AA = 'ACDEFGHIKLMNPQRSTVWY'

cgr = json.load(open(os.path.join(PROJECT, 'data', 'coupling_guided_rescue.json')))
TARGET_CK = 407
tkey = str(TARGET_CK)
if tkey not in cgr['targets']:
    raise SystemExit('FATAL chicken %d is not a target in coupling_guided_rescue.json'
                     % TARGET_CK)
T = cgr['targets'][tkey]
if abs(cgr['threshold'] - THRESHOLD) > 1e-12:
    raise SystemExit('FATAL the sweep used a threshold of %g, this notebook uses %g'
                     % (cgr['threshold'], THRESHOLD))

ranked = T['top_coupled']
if len(ranked) < N_POS:
    raise SystemExit('FATAL coupling_guided_rescue.json lists %d coupled positions for '
                     'chicken %d, this notebook needs %d'
                     % (len(ranked), TARGET_CK, N_POS))
CAND = [int(r['chicken_pos']) for r in ranked[:N_POS]]
if len(set(CAND)) != N_POS:
    raise SystemExit('FATAL the candidate list repeats a position: %s' % CAND)
if TARGET_CK in CAND:
    raise SystemExit('FATAL the target position is in its own candidate list')

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'primary_substitution_risk.tsv'))]
col = {k: i for i, k in enumerate(rows[0])}
tr = [r for r in rows[1:] if int(r[col['chicken_pos']]) == TARGET_CK]
if len(tr) != 1:
    raise SystemExit('FATAL chicken %d appears %d times in the step 1 risk table'
                     % (TARGET_CK, len(tr)))
tr = tr[0]
TARGET_CK_AA = tr[col['chicken_aa']]
TARGET_HU_AA = tr[col['human_aa']]
TARGET_HU_POS = int(tr[col['human_pos']])
P_BASELINE = float(tr[col['p_human_wt_background']])

if chicken[TARGET_CK - 1] != TARGET_CK_AA:
    raise SystemExit('FATAL chicken %d is %s in the sequence, %s in the table'
                     % (TARGET_CK, chicken[TARGET_CK - 1], TARGET_CK_AA))
if human[TARGET_HU_POS - 1] != TARGET_HU_AA:
    raise SystemExit('FATAL human %d is %s in the sequence, %s in the table'
                     % (TARGET_HU_POS, human[TARGET_HU_POS - 1], TARGET_HU_AA))
if P_BASELINE >= THRESHOLD:
    raise SystemExit('FATAL baseline %.3e is not below the threshold' % P_BASELINE)
for j in CAND:
    if not 1 <= j <= len(chicken):
        raise SystemExit('FATAL candidate position %d is outside the chicken sequence' % j)

pairs = [(CAND[a], CAND[b]) for a in range(N_POS) for b in range(a + 1, N_POS)]
doubles = [(i, x, j, y) for (i, j) in pairs
           for x in AA if x != chicken[i - 1]
           for y in AA if y != chicken[j - 1]]
singles = [(j, y) for j in CAND for y in AA if y != chicken[j - 1]]

log('target %s%d to %s, human %s%d, baseline %.3e'
    % (TARGET_CK_AA, TARGET_CK, TARGET_HU_AA, TARGET_HU_AA, TARGET_HU_POS, P_BASELINE))
log('candidate positions %s' % CAND)
log('pairs %d, double variants %d, single variants %d'
    % (len(pairs), len(doubles), len(singles)))


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
SCHEMA = 'v4-double-rescue-407'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()

base_ids = tok(chicken, return_tensors='pt')['input_ids'][0]
offset = 1
READ_IDX = TARGET_CK - 1 + offset
READ_TOK = tok.convert_tokens_to_ids(TARGET_HU_AA)
if READ_TOK is None or READ_TOK < 0:
    raise SystemExit('FATAL the tokenizer has no id for %s' % TARGET_HU_AA)
if int(base_ids[READ_IDX]) != tok.convert_tokens_to_ids(TARGET_CK_AA):
    raise SystemExit('FATAL the token at the read index is not the chicken residue, so '
                     'the offset is wrong')

AAID = {y: tok.convert_tokens_to_ids(y) for y in AA}
if any(v is None or v < 0 for v in AAID.values()):
    raise SystemExit('FATAL the tokenizer has no id for one of the amino acids')


def score(variants, ckpt, header, key_of, apply_of):
    done = set()
    if os.path.exists(ckpt) and os.path.getsize(ckpt) > 0:
        first = open(ckpt).readline().rstrip(chr(10))
        if first != header:
            raise SystemExit('FATAL checkpoint %s was written by a different schema; '
                             'delete it and rerun' % ckpt)
        for line in open(ckpt):
            p = line.rstrip(chr(10)).split(chr(9))
            if p[0] == 'row':
                done.add(key_of(p))
        log('resuming %s, already scored %d' % (os.path.basename(ckpt), len(done)))
    todo = [v for v in variants if key_of(['row'] + [str(x) for x in v]) not in done]
    log('%s: %d to score of %d' % (os.path.basename(ckpt), len(todo), len(variants)))
    t0 = time.time()
    with open(ckpt, 'a') as fh:
        if not done:
            fh.write(header + chr(10))
            fh.flush()
        for b in range(0, len(todo), BATCH):
            chunk = todo[b:b + BATCH]
            batch = base_ids.repeat(len(chunk), 1)
            for k, v in enumerate(chunk):
                apply_of(batch, k, v)
            with torch.no_grad():
                logits = model(batch.cuda()).logits
            sel = torch.softmax(logits[:, READ_IDX, :].float(), dim=-1)
            for k, v in enumerate(chunk):
                p = float(sel[k, READ_TOK])
                fh.write(chr(9).join(['row'] + [str(x) for x in v]
                                     + ['%.6e' % p]) + chr(10))
            fh.flush()
            if (b // BATCH) % 25 == 0:
                log('  %d of %d, %.1f min elapsed'
                    % (b + len(chunk), len(todo), (time.time() - t0) / 60))
    log('%s complete, %.1f min' % (os.path.basename(ckpt), (time.time() - t0) / 60))


def apply_double(batch, k, v):
    i, x, j, y = v
    batch[k, i - 1 + offset] = AAID[x]
    batch[k, j - 1 + offset] = AAID[y]


def apply_single(batch, k, v):
    j, y = v
    batch[k, j - 1 + offset] = AAID[y]


SING_CKPT = os.path.join(OUT, 'singles_partial.tsv')
SING_HEADER = chr(9).join(['schema_' + SCHEMA, 'pos', 'aa', 'p_target'])
score(singles, SING_CKPT, SING_HEADER,
      lambda p: (int(p[1]), p[2]), apply_single)

DBL_CKPT = os.path.join(OUT, 'doubles_partial.tsv')
DBL_HEADER = chr(9).join(['schema_' + SCHEMA, 'pos_i', 'aa_i', 'pos_j', 'aa_j',
                          'p_target'])
score(doubles, DBL_CKPT, DBL_HEADER,
      lambda p: (int(p[1]), p[2], int(p[3]), p[4]), apply_double)


In [ ]:
sing = {}
for line in open(SING_CKPT):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        sing[(int(p[1]), p[2])] = float(p[3])
if len(sing) != len(singles):
    raise SystemExit('FATAL %d singles in the checkpoint, %d expected'
                     % (len(sing), len(singles)))

dbl = {}
for line in open(DBL_CKPT):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        dbl[(int(p[1]), p[2], int(p[3]), p[4])] = float(p[5])
if len(dbl) != len(doubles):
    raise SystemExit('FATAL %d doubles in the checkpoint, %d expected'
                     % (len(dbl), len(doubles)))

prev = {}
scan = open(os.path.join(PROJECT, 'data', 'esm2_step2', 'rescue_scan.tsv'))
head = scan.readline().rstrip(chr(10)).split(chr(9))
sc = {k: i for i, k in enumerate(head)}
for line in scan:
    r = line.rstrip(chr(10)).split(chr(9))
    if int(r[sc['target_pos']]) != TARGET_CK:
        continue
    prev[(int(r[sc['rescuer_pos']]), r[sc['rescuer_to']])] = float(r[sc['p_target']])
if not prev:
    raise SystemExit('FATAL the step 2 scan holds no rows for target %d' % TARGET_CK)

TOL = 5e-2
worst = (0.0, None)
checked = 0
for k, v in sing.items():
    if k not in prev:
        raise SystemExit('FATAL %s%d is not in the step 2 scan, so the two runs cannot '
                         'be compared' % (k[1], k[0]))
    a, b = v, prev[k]
    rel = abs(a - b) / max(a, b, 1e-30)
    checked += 1
    if rel > worst[0]:
        worst = (rel, (k, a, b))
if worst[0] > TOL:
    k, a, b = worst[1]
    raise SystemExit('FATAL the rescored singles disagree with step 2: %s%d gives %.6e '
                     'here and %.6e there, a relative difference of %.2e above the '
                     'tolerance of %.0e. The two runs are not on one scale.'
                     % (k[1], k[0], a, b, worst[0], TOL))
log('singles rechecked against step 2: %d compared, worst relative difference %.2e'
    % (checked, worst[0]))

best_prev = max(((k, prev[k]) for k in sing), key=lambda kv: kv[1])
best_here = max(sing.items(), key=lambda kv: kv[1])
if best_prev[0] != best_here[0]:
    raise SystemExit('FATAL the strongest single at the candidate positions is %s%d here '
                     'and %s%d in step 2, so the two runs do not order the same way'
                     % (best_here[0][1], best_here[0][0],
                        best_prev[0][1], best_prev[0][0]))
log('strongest single agrees with step 2: %s%d at %.6e here, %.6e there'
    % (best_here[0][1], best_here[0][0], best_here[1], best_prev[1]))

best_single = max(sing.items(), key=lambda kv: kv[1])
rows_out = []
for (i, x, j, y), p in dbl.items():
    pi, pj = sing[(i, x)], sing[(j, y)]
    rows_out.append({'pos_i': i, 'from_i': chicken[i - 1], 'to_i': x,
                     'pos_j': j, 'from_j': chicken[j - 1], 'to_j': y,
                     'p_target': p, 'p_single_i': pi, 'p_single_j': pj,
                     'best_single_of_pair': max(pi, pj),
                     'fold_vs_baseline': p / P_BASELINE,
                     'fold_vs_best_single_of_pair': p / max(pi, pj),
                     'crosses': p >= THRESHOLD})
rows_out.sort(key=lambda r: -r['p_target'])

FINAL = os.path.join(OUT, 'double_rescue.tsv')
cols = ['pos_i', 'from_i', 'to_i', 'pos_j', 'from_j', 'to_j', 'p_target',
        'p_single_i', 'p_single_j', 'best_single_of_pair', 'fold_vs_baseline',
        'fold_vs_best_single_of_pair', 'crosses']
with open(FINAL, 'w') as fh:
    fh.write(chr(9).join(cols) + chr(10))
    for r in rows_out:
        fh.write(chr(9).join(
            ('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
            for c in cols) + chr(10))

SFINAL = os.path.join(OUT, 'singles_recheck.tsv')
with open(SFINAL, 'w') as fh:
    fh.write(chr(9).join(['pos', 'from', 'to', 'p_target', 'p_target_step2']) + chr(10))
    for (j, y), v in sorted(sing.items()):
        fh.write(chr(9).join([str(j), chicken[j - 1], y, '%.6e' % v,
                              '%.6e' % prev[(j, y)]]) + chr(10))

crossing = [r for r in rows_out if r['crosses']]
summary = {'schema': SCHEMA, 'model': MODEL, 'chicken_sha256': CHICKEN_SHA,
           'target_chicken_pos': TARGET_CK, 'target_chicken_aa': TARGET_CK_AA,
           'target_human_pos': TARGET_HU_POS, 'target_human_aa': TARGET_HU_AA,
           'threshold': THRESHOLD, 'p_baseline': P_BASELINE,
           'candidate_positions': CAND, 'n_pairs': len(pairs),
           'n_doubles': len(doubles), 'n_singles': len(singles),
           'singles_compared_to_step2': checked,
           'worst_relative_difference_vs_step2': worst[0],
           'best_single': {'pos': best_single[0][0], 'to': best_single[0][1],
                           'p': best_single[1]},
           'best_double': rows_out[0] if rows_out else None,
           'n_doubles_crossing': len(crossing),
           'n_doubles_above_best_single': sum(
               1 for r in rows_out if r['p_target'] > best_single[1]),
           'max_fold_vs_best_single_of_pair': max(
               (r['fold_vs_best_single_of_pair'] for r in rows_out), default=None)}
with open(os.path.join(OUT, 'double_rescue_summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)

log('wrote %s, %s and the summary' % (os.path.basename(FINAL), os.path.basename(SFINAL)))
log('doubles crossing %.0e: %d of %d' % (THRESHOLD, len(crossing), len(rows_out)))
if rows_out:
    r = rows_out[0]
    log('best double %s%d%s with %s%d%s at %.6e, %.1f-fold over baseline and %.2f-fold '
        'over the better single of the pair'
        % (r['from_i'], r['pos_i'], r['to_i'], r['from_j'], r['pos_j'], r['to_j'],
           r['p_target'], r['fold_vs_baseline'], r['fold_vs_best_single_of_pair']))
log('best single here %s%d at %.6e'
    % (best_single[0][1], best_single[0][0], best_single[1]))
log('copy esm2_step4/ from Drive into data/ and run codes/16_double_rescue_read.py')
